# Indic PEFT study — run everything on a free Colab T4

1. **Runtime → Change runtime type → T4 GPU**
2. Run the cells top to bottom. Results are saved to Google Drive, so if Colab disconnects, re-run everything: finished runs are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/indic-peft-study/results'

In [ ]:
!git clone https://github.com/MEHTABSINGH1/indic-peft-study.git /content/indic-peft-study || (cd /content/indic-peft-study && git pull)
%cd /content/indic-peft-study
!pip -q install -r requirements.txt
!nvidia-smi -L

In [ ]:
!python -m pytest -q   # sanity check, ~1 min

## RQ1 + RQ3 — main sweep (90 runs, ~2 h)
If you have two Colab accounts or a Kaggle notebook, split it: `--shard 0/2` here and `--shard 1/2` there.

In [ ]:
!python scripts/sweep.py configs/main.yaml --out-dir $RESULTS/runs

## RQ2 — ablations (~1 h each)

In [ ]:
!python scripts/sweep.py configs/ablation_rank.yaml --out-dir $RESULTS/runs

In [ ]:
!python scripts/sweep.py configs/ablation_size.yaml --out-dir $RESULTS/runs

## RQ4 (optional) — Gemma-2-2B + QLoRA
First accept the licence at https://huggingface.co/google/gemma-2-2b, then log in.

In [ ]:
# from huggingface_hub import login; login()
# !python scripts/sweep.py configs/llm_qlora.yaml --out-dir $RESULTS/runs

## Tables, figures, error analysis

In [ ]:
!python scripts/aggregate.py --runs $RESULTS/runs --out $RESULTS
!python scripts/error_analysis.py --runs $RESULTS/runs --out $RESULTS/error_analysis --model muril-base-cased --method full --train-lang hi --test-langs hi pa ur

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{RESULTS}/RESULTS.md', encoding='utf-8').read()))
import glob
for f in sorted(glob.glob(f'{RESULTS}/figures/*.png')): display(Image(f, width=480))

Finally, copy `results/` from Drive into the repo and push, so the README links work.